# 🔁 Lab W3-3 — ETL Pipeline ที่ทนต่อความจริง

**สัปดาห์ที่ 3 — Data Management & Data Warehouse**

ใช้คู่กับสื่อจำลอง **ETL Pipeline Sim** (`/sims/etl-pipeline`)

## สิ่งที่จะได้เรียนรู้
1. เขียน **incremental load** ที่เป็น **idempotent** — รันซ้ำแล้วผลไม่เปลี่ยน
2. ตรวจจับ **schema drift** อัตโนมัติแทนที่จะรู้ตัวตอนรายงานผิด
3. จัดการ **late-arriving data** ด้วยการพาร์ทิชันตามวันที่เกิดเหตุการณ์จริง

## ข้อมูล
แฟ้มรายวัน 30 ไฟล์ เดือนกันยายน 2025 — มีเหตุการณ์ 4 อย่างที่ไม่มีใครแจ้งล่วงหน้า

In [ ]:
import pandas as pd

BASE = ("https://raw.githubusercontent.com/babankbro/ksu-dss-course/"
        "master/datasets/week03/daily_batches/")
DAYS = pd.date_range("2025-09-01", "2025-09-30").strftime("%Y-%m-%d").tolist()

files = {}
for d in DAYS:
    try:
        files[d] = pd.read_csv(BASE + f"batch_{d}.csv")
    except pd.errors.EmptyDataError:
        files[d] = pd.DataFrame()

print(f"โหลดแฟ้ม {len(files)} ไฟล์")
for d in ["2025-09-01", "2025-09-11", "2025-09-12", "2025-09-27"]:
    print(f"  {d}: {len(files[d]):>3} แถว | คอลัมน์ = {list(files[d].columns)}")

> **สังเกตตั้งแต่ตอนนี้** แฟ้มวันที่ 12 มีคอลัมน์ไม่เหมือนวันที่ 11
> ในระบบจริงจะไม่มีใครมาบอก — ท่อข้อมูลต้องตรวจเจอเอง

### 🧑‍💻 งานที่ 1 — ตรวจจับ schema drift
เขียนโค้ดที่เดินผ่านทุกแฟ้มแล้วรายงานว่า
1. วันใดที่ชุดคอลัมน์เปลี่ยนไปจากวันก่อนหน้า
2. คอลัมน์ใดหายไป และคอลัมน์ใดเพิ่มเข้ามา
3. วันใดที่แฟ้มว่าง (ต้องแยกให้ออกจาก "ท่อข้อมูลล้มเหลว")

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 2 — ท่อข้อมูลที่รับมือ schema drift ได้

### 🧑‍💻 งานที่ 2
เขียนฟังก์ชัน `normalize(df)` ที่จับคู่ชื่อคอลัมน์ให้เป็นมาตรฐานเดียว
โดยรองรับทั้งสองรุ่นของแฟ้ม (`amount` และ `net_amount`)

**ข้อควรระวัง** ห้ามใช้วิธี "ถ้าไม่มีคอลัมน์ก็ให้เป็น 0" เด็ดขาด —
นั่นคือความล้มเหลวที่เงียบที่สุด งานจะไม่ล้ม แต่ยอดเงินหายไปทั้งเดือน
ให้ `raise` ออกมาแทนถ้าจับคู่ไม่ได้

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 3 — Incremental load ที่ idempotent

นิยาม: รันท่อข้อมูลชุดเดิมซ้ำกี่ครั้งก็ได้ ผลลัพธ์ในคลังต้องเหมือนเดิมทุกครั้ง

### 🧑‍💻 งานที่ 3
เขียนฟังก์ชัน `load(warehouse, batch, load_date, partition_by)` ที่
* ไม่โหลดแถวที่ `txn_id` มีอยู่แล้ว (idempotency)
* รองรับการพาร์ทิชันสองแบบ: ตามวันที่ของแฟ้ม หรือ ตามวันที่เกิดธุรกรรมจริง

แล้วรันครบ 30 วัน 2 รอบ พิสูจน์ว่ารอบที่สองไม่เพิ่มแถวใดเลย

*เป้าหมาย: 1,549 แถว · ยอดรวม 1,822,503.00 บาท*

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 4 — Late-arriving data

แฟ้มวันที่ 18 มีธุรกรรมของวันที่ 16 และ 17 ปนมาด้วย
การเลือกวิธีพาร์ทิชันจะตัดสินว่า **รายงานย้อนหลังถูกหรือผิด**

### 🧑‍💻 งานที่ 4
1. หาว่ามีกี่แถวที่ `txn_date` ไม่ตรงกับวันที่ของแฟ้มที่บรรจุมัน
2. เปรียบเทียบยอดขายของวันที่ 16, 17 และ 18 ระหว่างการพาร์ทิชันสองแบบ
3. ตอบว่าถ้าผู้บริหารดูรายงานวันที่ 17 ในเช้าวันที่ 18 แล้วมาดูซ้ำอีกครั้งในวันที่ 19
   ตัวเลขควรเปลี่ยนหรือไม่ และควรสื่อสารกับผู้ใช้อย่างไร

In [ ]:
# เขียนโค้ดของคุณที่นี่


## ส่วนที่ 5 — ตาราง etl_audit

### 🧑‍💻 งานที่ 5
สร้าง `etl_audit` ที่บันทึกรายวัน: แถวในแฟ้ม · แถวที่โหลดเข้า · แถวที่ข้าม ·
ยอดเงินของวันนั้น · สถานะ (`ok` / `empty` / `schema_drift` / `duplicate_file`)

แล้วใช้ตารางนี้ตอบว่า **วันใดที่ควรมีการแจ้งเตือนไปยังทีมข้อมูล**

In [ ]:
# เขียนโค้ดของคุณที่นี่


> **ประเด็นสำคัญที่สุดของ Lab นี้**
>
> ในทั้ง 4 เหตุการณ์ ท่อข้อมูลรายงานว่า **ทำงานสำเร็จทุกวัน** ไม่มี exception ใดถูกโยนออกมา
> ความเสียหายทั้งหมดเกิดขึ้นเงียบๆ และจะถูกค้นพบก็ต่อเมื่อมีคนสังเกตว่าตัวเลขแปลก
> ซึ่งอาจเป็นเวลาหลายสัปดาห์ให้หลัง
>
> นี่คือเหตุผลที่ **data observability** (etl_audit, freshness check, row-count anomaly)
> เป็นองค์ประกอบบังคับของสถาปัตยกรรมคลังข้อมูล ไม่ใช่ของแถม

---
## ✅ เกณฑ์การส่งงาน

| องค์ประกอบ | คะแนน |
|---|:--:|
| งานที่ 1 — ตรวจจับ schema drift และแฟ้มว่างได้ | 2 |
| งานที่ 2 — normalize ที่ล้มดังเมื่อจับคู่ไม่ได้ | 2 |
| งานที่ 3 — incremental load ที่พิสูจน์ idempotency ได้ | 3 |
| งานที่ 4 — วิเคราะห์ late-arriving data และผลต่อรายงาน | 3 |
| งานที่ 5 — etl_audit และการระบุวันที่ต้องแจ้งเตือน | 2 |
| **รวม** | **12** |

> 💡 ผลลัพธ์ที่ถูกต้อง: **1,549 แถว** · ยอดรวม **1,822,503.00 บาท**